# RosettaFold 3 Modeling
## **Purpose:** To model the CRBN-MGD-ZBTB11/IKZF1 ternary complexes using RF3 


### **Package Installs**

In [1]:
import sys

print(f"Target Environment Path: {sys.prefix}")

print("1. Installing core physics via Conda...")
!conda install --prefix {sys.prefix} -c conda-forge openmmforcefields mdtraj gemmi pydantic -y -q

print("1.5. Installing openMM for appropriate CUDA via Conda...")
!conda install --prefix {sys.prefix} -c conda-forge openmm "cuda-version=13.0" -y

print("2. Cloning OpenFF directly via Git to preserve versioning data...")
!{sys.executable} -m pip install git+https://github.com/openforcefield/openff-toolkit.git --no-cache-dir --upgrade --no-deps -q
!{sys.executable} -m pip install git+https://github.com/openforcefield/openff-interchange.git --no-cache-dir --upgrade --no-deps -q

print("3. Installing ML tools via Pip...")
!{sys.executable} -m pip install biopython pdbfixer torchmetrics "atomworks[ml]" rc-foundry[rf3] --no-cache-dir -q

print("✅ Setup complete!")

Target Environment Path: /home/ntran/miniconda3/envs/physics_env
1. Installing core physics via Conda...
Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.

1.5. Installing openMM for appropriate CUDA via Conda...
Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.

2. Cloning OpenFF directly via Git to preserve versioning data...
3. Installing ML tools via Pip...
✅ Setup complete!


### **Platform Validation**

In [2]:
from openmm import Platform

speed = Platform.getPlatformByName('CUDA').getSpeed()
print(f"CUDA Platform Speed: {speed}")
print(f"Successful connection to GPU!")

CUDA Platform Speed: 100.0
Successful connection to GPU!


In [3]:
import os
import urllib.request

url = "http://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt"
filename = "rf3_foundry_01_24_latest_remapped.ckpt"

if os.path.exists(filename):
    print(f"'{filename}' already exists! Skipping download.")
else:
    print("Downloading RF3 weights... this might take a few minutes...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete!")

'rf3_foundry_01_24_latest_remapped.ckpt' already exists! Skipping download.


### **RF3 Batch Processing Pipeline (Double Templating) - CRBN:MGD:ZBTB11**

In [10]:
import json
import subprocess
import pandas as pd
import numpy as np
import os
from Bio.PDB.MMCIFParser import MMCIFParser


# 1. CLEAN AND REFORMAT INPUT PDBS
def clean_and_rechain_pdb(input_file, output_file, protein_chain, hetatm_chain):
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        for line in f_in:
            if "HOH" in line or "WAT" in line:
                continue
                
            if line.startswith("ATOM  "):
                line = line[:21] + protein_chain + line[22:]
                f_out.write(line)
            elif line.startswith("HETATM"):
                line = line[:21] + hetatm_chain + line[22:]
                f_out.write(line)
            elif line.startswith("TER") or line.startswith("END"):
                f_out.write(line)

clean_and_rechain_pdb("crbn.pdb", "crbn_clean.pdb", protein_chain="A", hetatm_chain="Y")      # Labeling CRBN as chain A, any detected heteroatoms as chain Y
clean_and_rechain_pdb("zbtb11.pdb", "zbtb11_clean.pdb", protein_chain="B", hetatm_chain="Z")  # Labeling ZBTB11 as chain B, any detected heteroatoms as chain Z


# 2. SET CONSTANTS & PATHS
CRBN_PDB_PATH = "./crbn_clean.pdb"       
ZBTB11_PDB_PATH = "./zbtb11_clean.pdb"   
INPUT_CSV_PATH = "mgd_list.csv"          
OUTPUT_DIR = "./rf3_outputs_zbtb11"             
BATCH_JSON_PATH = "mgd_batch_zbtb11.json"       
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 3. DEFINE YOUR MGD LIBRARY FROM CSV
print(f"Loading molecules from {INPUT_CSV_PATH}...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    
    if 'Molecule' not in df.columns or 'SMILES' not in df.columns:
        raise ValueError("The input CSV must contain 'Molecule' and 'SMILES' columns.")
    
    mgd_library = list(zip(df['Molecule'], df['SMILES']))
    print(f"Successfully loaded {len(mgd_library)} compounds.")
    
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_CSV_PATH}. Please check the file path.")
    exit()


# 4. GENERATE THE BATCH JSON (DOUBLE TEMPLATING)
batch_config = []
for cmpd_id, smiles in mgd_library:
    batch_config.append({
        "name": str(cmpd_id),
        "components": [
            {
                "path": CRBN_PDB_PATH
            },
            {
                "path": ZBTB11_PDB_PATH
            },
            {
                "smiles": str(smiles)
            }
        ],
        "template_selection": ["A", "B", "Y", "Z"] 
    })

with open(BATCH_JSON_PATH, "w") as f:
    json.dump(batch_config, f, indent=4)

print(f"Saved {len(mgd_library)} configurations to {BATCH_JSON_PATH}.")


# 5. EXECUTE RF3 BATCH INFERENCE
print("Executing RF3 inference for ZBTB11...")
seeds_to_run = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

for current_seed in seeds_to_run:
    print(f"\n--- Starting run for Seed {current_seed} ---")
    
    seed_out_dir = os.path.join(OUTPUT_DIR, f"seed_{current_seed}")
    os.makedirs(seed_out_dir, exist_ok=True)
    
    cmd = [
        "rf3", "fold",
        f"inputs={BATCH_JSON_PATH}",
        f"out_dir={seed_out_dir}",
        "ckpt_path=./rf3_foundry_01_24_latest_remapped.ckpt",
        "early_stopping_plddt_threshold=0.5",
        
        # MODELING OPTIMIZATIONS
        "diffusion_batch_size=1",  # Generate 1 structure per model seed instead of default 5
        "num_steps=50",            # Run 50 diffusion steps instead of default 200
        "n_recycles=5",            # Run 5 cycles through the GNN instead of default 10
        f"seed={current_seed}"     # Model based on 10 different initial model seeds instead of default 1
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Seed {current_seed} inference complete!")
        
    except subprocess.CalledProcessError as e:
        print(f"\nRF3 CRASHED on Seed {current_seed}! Here is the internal error log:")
        print("-" * 40)
        print(e.stderr)
        print("-" * 40)


# 6. EXTRACT FEATURES FOR RIDGE AND CHEMPROP
print("\nExtracting confidence scores...")
extracted_features = []

for cmpd_id, smiles in mgd_library:
    for current_seed in seeds_to_run:
        cmpd_dir = os.path.join(OUTPUT_DIR, f"seed_{current_seed}", str(cmpd_id))
        
        summary_file = os.path.join(cmpd_dir, f"{cmpd_id}_summary_confidences.json")
        score_file = os.path.join(cmpd_dir, f"{cmpd_id}.score")
        
        if os.path.exists(summary_file):
            with open(summary_file, 'r') as f:
                summary = json.load(f)
                
                extracted_features.append({
                    "Compound_ID": cmpd_id,
                    "SMILES": smiles,
                    "Seed": current_seed,
                    "ipTM_zbtb11": summary.get('iptm', None),
                    "pTM_zbtb11": summary.get('ptm', None),
                    "pLDDT_zbtb11": summary.get('overall_plddt', None),
                    "Ranking_Score_zbtb11": summary.get('ranking_score', None),
                    "Early_Stopped_zbtb11": False 
                })
                
        elif os.path.exists(score_file):
            with open(score_file, 'r') as f:
                score_data = json.load(f)
                extracted_features.append({
                    "Compound_ID": cmpd_id,
                    "SMILES": smiles,
                    "Seed": current_seed,
                    "ipTM_zbtb11": 0.0, 
                    "pTM_zbtb11": 0.0,
                    "pLDDT_zbtb11": score_data.get('overall_plddt', None), 
                    "Ranking_Score_zbtb11": 0.0,
                    "Early_Stopped_zbtb11": True
                })
                
        else:
            extracted_features.append({
                "Compound_ID": cmpd_id,
                "SMILES": smiles,
                "Seed": current_seed,
                "ipTM_zbtb11": None,
                "pTM_zbtb11": None,
                "pLDDT_zbtb11": None,
                "Ranking_Score_zbtb11": None,
                "Early_Stopped_zbtb11": None
            })

features_df = pd.DataFrame(extracted_features)
total_trajectories = len(features_df)
successes = len(features_df[features_df['Early_Stopped_zbtb11'] == False])
print(f"Total Trajectories Processed: {total_trajectories} ({len(mgd_library)} compounds x {len(seeds_to_run)} seeds)")
print(f"Successful Binders (Full Fold): {successes}")


# 7. VALIDATE DEGRON-LIGAND DISTANCE IN ALL TRAJECTORIES
print("\nValidating Degron-Ligand distances across all generated models...")

def check_degron_ligand_distance_cif(cif_path, degron_chain='B', ligand_resname='L:0'):
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure('complex', cif_path)
    except Exception as e:
        return None, f"Parse Error: {e}"

    degron_coords = []
    ligand_coords = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if chain.id == degron_chain:
                    for atom in residue:
                        degron_coords.append(atom.coord)
                
                if residue.resname == ligand_resname:
                    for atom in residue:
                        if atom.element != 'ZN':
                            ligand_coords.append(atom.coord)

    if not degron_coords:
        return None, f"Degron chain '{degron_chain}' not found"
    if not ligand_coords:
        return None, f"Ligand '{ligand_resname}' not found"

    degron_coords = np.array(degron_coords)
    ligand_coords = np.array(ligand_coords)

    diff = degron_coords[:, np.newaxis, :] - ligand_coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff**2, axis=2))
    
    min_dist = np.min(dist_matrix)
    return min_dist, "Success"


validation_results = []
for _, row in features_df.iterrows():
    cmpd_id = row['Compound_ID']
    seed = row['Seed']
    
    cif_file = os.path.join(OUTPUT_DIR, f"seed_{seed}", str(cmpd_id), f"{cmpd_id}_model.cif")
    
    if os.path.exists(cif_file):
        dist, status = check_degron_ligand_distance_cif(cif_file)
        validation_results.append({
            "Compound_ID": cmpd_id,
            "Seed": seed,
            "Min_Dist_Angstrom_zbtb11": dist,
            "Dist_Status_zbtb11": status,
            "Flagged_Greater_Than_10A_zbtb11": (dist > 10.0) if dist is not None else True 
        })
    else:
        validation_results.append({
            "Compound_ID": cmpd_id,
            "Seed": seed,
            "Min_Dist_Angstrom_zbtb11": None,
            "Dist_Status_zbtb11": "File not found",
            "Flagged_Greater_Than_10A_zbtb11": True
        })

dist_df = pd.DataFrame(validation_results)

final_combined_df = pd.merge(features_df, dist_df, on=['Compound_ID', 'Seed'], how='left')

successful_complexes = final_combined_df[final_combined_df['Flagged_Greater_Than_10A_zbtb11'] == False]
failed_complexes = final_combined_df[final_combined_df['Flagged_Greater_Than_10A_zbtb11'] == True]

print(f"\nDistance Check Summary:")
print(f"{len(successful_complexes)} trajectories successfully bridged the interface (< 10A).")
print(f"{len(failed_complexes)} trajectories failed to recruit the degron (> 10A).")


# 8. CALCULATE SUCCESS FRACTION PER COMPOUND & MERGE
print("\nCalculating success fractions for each Compound_ID...")

success_fraction_df = final_combined_df.groupby('Compound_ID')['Flagged_Greater_Than_10A_zbtb11'].apply(
    lambda x: (x == False).mean()
).reset_index()

success_fraction_df.rename(columns={'Flagged_Greater_Than_10A_zbtb11': 'Fraction_Success_Under_10A_zbtb11'}, inplace=True)

summary_output_csv = "rf3_compound_success_fractions_zbtb11.csv"
success_fraction_df.to_csv(summary_output_csv, index=False)

print(f"\nFraction of successful complexes (< 10A) per Compound_ID:")
print(success_fraction_df.head())
print(f"Compound summary saved to: {summary_output_csv}")

final_combined_df = pd.merge(final_combined_df, success_fraction_df, on='Compound_ID', how='left')

final_output_csv = "rf3_all_features_and_distances_zbtb11.csv"
final_combined_df.to_csv(final_output_csv, index=False)

print(f"Combined data (with success fractions appended) saved to: {final_output_csv}")
print(f"\nPipeline complete! All features and validations successfully saved.")


Validating Degron-Ligand distances across all generated models...

Distance Check Summary:
233 trajectories successfully bridged the interface (< 10A).
647 trajectories failed to recruit the degron (> 10A).

Calculating success fractions for each Compound_ID...

Fraction of successful complexes (< 10A) per Compound_ID:
    Compound_ID  Fraction_Success_Under_10A
0    ALV-05-184                         0.1
1  BML-01-020-L                         0.0
2  BML-01-020-U                         0.6
3    JWJ-01-200                         0.0
4    JWJ-01-203                         0.0
Compound summary saved to: rf3_compound_success_fractions.csv
Combined data (with success fractions appended) saved to: rf3_all_features_and_distances.csv

Pipeline complete! All features and validations successfully saved.


### **Model Visualization - CRBN:MGD:IKZF1**

In [5]:
from atomworks.io.utils.visualize import view
from atomworks.io.utils.io_utils import load_any

# 1. Define the path to your generated file
model_path = "./rf3_outputs_zbtb11/seed_42/ALV-05-184/ALV-05-184_model.cif"

# 2. Parse the CIF file into a 3D structural object (AtomArray)
parsed_structure = load_any(model_path)

# 3. Render the interactive 3D view
view(parsed_structure)

Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
AtomArrayStack is not supported; using the first model.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### **Energy Minimization and Scoring - CRBN:MGD:ZBTB11**

In [12]:
import os
os.environ["OPENMM_CPU_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import traceback
import pandas as pd
import gemmi
import multiprocessing as mp
from openmm import XmlSerializer, LangevinMiddleIntegrator, Platform, NonbondedForce
from openmm.app import PDBFile, Modeller, ForceField, Simulation, NoCutoff, HBonds, Element
from openmm.unit import kelvin, picosecond, kilojoule_per_mole, angstrom, elementary_charge, nanometer
from openff.toolkit import Molecule
from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem


# ==========================================
# PHASE 1: CPU PREPARATION & SERIALIZATION
# ==========================================
def prep_complex_cpu(task_info):
    index, row = task_info
    cmpd_id = row["Compound_ID"]
    smiles = row["SMILES"]
    seed = row["Seed"]
    
    base_dir = os.path.join("rf3_outputs_zbtb11", f"seed_{seed}", str(cmpd_id))
    cif_path = os.path.join(base_dir, f"{cmpd_id}_model.cif")
    pdb_path = os.path.join(base_dir, f"{cmpd_id}_converted.pdb")
    
    system_xml_path = os.path.join(base_dir, f"{cmpd_id}_system.xml")
    ready_pdb_path = os.path.join(base_dir, f"{cmpd_id}_ready.pdb")
    
    if not os.path.exists(cif_path):
        return index, False

    if os.path.exists(system_xml_path) and os.path.exists(ready_pdb_path):
        return index, True

    try:
        # 1. CONVERT CIF TO PDB
        if not os.path.exists(pdb_path):
            doc = gemmi.cif.read_file(cif_path)
            st = gemmi.make_structure_from_block(doc.sole_block())
            st.write_pdb(pdb_path)
            
        # 2. RENAMING & SEPARATION
        prot_pdb_path = os.path.join(base_dir, f"{cmpd_id}_prot.pdb")
        lig_pdb_path = os.path.join(base_dir, f"{cmpd_id}_lig.pdb")
        standard_res = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 
                        'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 
                        'TYR', 'VAL', 'HOH', 'ACE', 'NME', 'MG', 'CA', 'NA', 'CL', 'K', 'ZN'}
        
        with open(pdb_path, 'r') as f: 
            lines = f.readlines()
            
        with open(prot_pdb_path, 'w') as f_prot, open(lig_pdb_path, 'w') as f_lig:
            for line in lines:
                line = line.replace('L:0', 'LIG')
                if line.startswith(('ATOM', 'HETATM')):
                    resname = line[17:21].strip()
                    if resname in standard_res:
                        f_prot.write(line)
                    else:
                        f_lig.write(line)
                elif line.startswith(('HET', 'HETNAM', 'HETSYN', 'FORMUL', 'CONECT')):
                    if 'LIG' in line: f_lig.write(line)
                elif line.startswith('END'):
                    f_prot.write(line); f_lig.write(line)
                else:
                    f_prot.write(line)

        # 3. PREPARE OPENMM FORCEFIELD & FIX PROTEIN
        forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml', 'implicit/gbn2.xml')
        fixer = PDBFixer(filename=prot_pdb_path)
        zn_elem = Element.getBySymbol('Zn')
        for atom in fixer.topology.atoms():
            if atom.residue.name == 'ZN':
                atom.name = 'Zn'; atom.element = zn_elem

        fixer.findMissingResidues()
        fixer.missingResidues = {} 
        fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms() 

        modeller = Modeller(fixer.topology, fixer.positions)
        modeller.addHydrogens(forcefield, pH=7.0)

        # 4. PROCESS LIGAND
        lig_rdkit = Chem.MolFromPDBFile(lig_pdb_path, sanitize=False)
        frags = Chem.GetMolFrags(lig_rdkit, asMols=True)
        off_molecules = []
        for frag in frags:
            if frag.GetNumHeavyAtoms() > 3:
                template_mol = Chem.MolFromSmiles(smiles)
                frag = AllChem.AssignBondOrdersFromTemplate(template_mol, frag)
                Chem.SanitizeMol(frag)
                frag_h = Chem.AddHs(frag, addCoords=True)
                for atom in frag_h.GetAtoms():
                    mi = Chem.AtomPDBResidueInfo()
                    mi.SetResidueName("LIG"); mi.SetResidueNumber(1)
                    atom.SetMonomerInfo(mi)
                off_molecules.append(Molecule.from_rdkit(frag_h, allow_undefined_stereo=True))

        # 5. REGISTER GENERATOR & CREATE SYSTEM
        generator = SMIRNOFFTemplateGenerator(molecules=off_molecules) 
        forcefield.registerTemplateGenerator(generator.generator)
        for mol in off_molecules:
            modeller.add(mol.to_topology().to_openmm(), mol.conformers[0].to_openmm())

        system = forcefield.createSystem(modeller.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
        
        # 6. SERIALIZE TO DISK (The Magic Step)
        with open(system_xml_path, 'w') as f:
            f.write(XmlSerializer.serialize(system))
        
        with open(ready_pdb_path, 'w') as f:
            PDBFile.writeFile(modeller.topology, modeller.positions, f)

        print(f"🛠️ [{cmpd_id}] CPU Prep Complete! Saved to XML.")
        return index, True
        
    except Exception as e:
        print(f"❌ [{cmpd_id}] CPU PREP FAILED: {e}")
        return index, False


# ==========================================
# PHASE 2: GPU MINIMIZATION
# ==========================================
def minimize_complex_gpu(task_info):
    index, row = task_info
    cmpd_id = row["Compound_ID"]
    seed = row["Seed"]
    
    base_dir = os.path.join("rf3_outputs_zbtb11", f"seed_{seed}", str(cmpd_id))
    system_xml_path = os.path.join(base_dir, f"{cmpd_id}_system.xml")
    ready_pdb_path = os.path.join(base_dir, f"{cmpd_id}_ready.pdb")
    
    if not os.path.exists(system_xml_path) or not os.path.exists(ready_pdb_path):
        return index, None

    try:
        # 1. LOAD PRE-BUILT SYSTEM AND POSITIONS
        with open(system_xml_path, 'r') as f:
            system = XmlSerializer.deserialize(f.read())
        
        pdb = PDBFile(ready_pdb_path)
        
        # 2. SIMULATE AND MINIMIZE
        integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picosecond)
        simulation = Simulation(pdb.topology, system, integrator, Platform.getPlatformByName('CUDA'))
        simulation.context.setPositions(pdb.positions)
        
        simulation.minimizeEnergy(tolerance=10.0*kilojoule_per_mole/nanometer)
        
        state = simulation.context.getState(getEnergy=True)
        minimized_energy = state.getPotentialEnergy().value_in_unit(kilojoule_per_mole)
        
        print(f"✅ [{cmpd_id}] GPU Minimized: {minimized_energy:.2f} kJ/mol")

        # 3. FREE VRAM
        del simulation.context
        del simulation
        import gc; gc.collect()
        
        return index, minimized_energy
        
    except Exception as e:
        print(f"❌ [{cmpd_id}] GPU SPRINT FAILED: {e}")
        return index, None

# ==========================================
# MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    csv_path = "rf3_all_features_and_distances_zbtb11.csv"
    df = pd.read_csv(csv_path)
    if "OpenMM_Minimized_Energy_kJ_mol" not in df.columns:
        df["OpenMM_Minimized_Energy_kJ_mol"] = None

    successful_complexes = df[(df["Early_Stopped_zbtb11"] == False)]
    all_tasks = list(successful_complexes.iterrows())

    # --- EXECUTE PHASE 1 ---
    print(f"\n🚀 PHASE 1: Starting CPU Preparation for {len(all_tasks)} complexes...")
    with mp.Pool(processes=32, maxtasksperchild=1) as pool:
        prep_results = pool.map(prep_complex_cpu, all_tasks)

    ready_tasks = [task for task, result in zip(all_tasks, prep_results) if result[1] == True]

    # --- EXECUTE PHASE 2 ---
    print(f"\n⚡ PHASE 2: Starting GPU Minimization Sprint for {len(ready_tasks)} prepped complexes...")
    with mp.Pool(processes=6, maxtasksperchild=1) as pool:
        min_results = pool.map(minimize_complex_gpu, ready_tasks)

    # --- SAVE RESULTS ---
    for index, energy in min_results:
        if energy is not None:
            df.at[index, "OpenMM_Minimized_Energy_kJ_mol_zbtb11"] = energy

    output_csv = "rf3_features_with_thermodynamics_zbtb11.csv"
    df.to_csv(output_csv, index=False)
    print(f"\n✨ Pipeline finished! Results saved to {output_csv}")


✨ Pipeline finished! Results saved to rf3_features_with_thermodynamics.csv


### **RF3 Batch Processing Pipeline (Double Templating) - CRBN:MGD:IKZF1**

In [4]:
import json
import subprocess
import pandas as pd
import numpy as np
import os
from Bio.PDB.MMCIFParser import MMCIFParser


# 1. CLEAN AND REFORMAT INPUT PDBS
def clean_and_rechain_pdb(input_file, output_file, protein_chain, hetatm_chain):
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        for line in f_in:
            if "HOH" in line or "WAT" in line:
                continue
                
            if line.startswith("ATOM  "):
                line = line[:21] + protein_chain + line[22:]
                f_out.write(line)
            elif line.startswith("HETATM"):
                line = line[:21] + hetatm_chain + line[22:]
                f_out.write(line)
            elif line.startswith("TER") or line.startswith("END"):
                f_out.write(line)

clean_and_rechain_pdb("crbn.pdb", "crbn_clean.pdb", protein_chain="A", hetatm_chain="Y")      # Labeling CRBN as chain A, any detected heteroatoms as chain Y
clean_and_rechain_pdb("ikzf1.pdb", "ikzf1_clean.pdb", protein_chain="B", hetatm_chain="Z")    # Labeling IKZF1 as chain B, any detected heteroatoms as chain Z


# 2. SET CONSTANTS & PATHS
CRBN_PDB_PATH = "./crbn_clean.pdb"       
IKZF1_PDB_PATH = "./ikzf1_clean.pdb"   
INPUT_CSV_PATH = "mgd_list.csv"          
OUTPUT_DIR = "./rf3_outputs_ikzf1"             
BATCH_JSON_PATH = "mgd_batch_ikzf1.json"       
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 3. DEFINE YOUR MGD LIBRARY FROM CSV
print(f"Loading molecules from {INPUT_CSV_PATH}...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    
    if 'Molecule' not in df.columns or 'SMILES' not in df.columns:
        raise ValueError("The input CSV must contain 'Molecule' and 'SMILES' columns.")
    
    mgd_library = list(zip(df['Molecule'], df['SMILES']))
    print(f"Successfully loaded {len(mgd_library)} compounds.")
    
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_CSV_PATH}. Please check the file path.")
    exit()


# 4. GENERATE THE BATCH JSON (DOUBLE TEMPLATING)
batch_config = []
for cmpd_id, smiles in mgd_library:
    batch_config.append({
        "name": str(cmpd_id),
        "components": [
            {
                "path": CRBN_PDB_PATH
            },
            {
                "path": IKZF1_PDB_PATH
            },
            {
                "smiles": str(smiles)
            }
        ],
        "template_selection": ["A", "B", "Y", "Z"] 
    })

with open(BATCH_JSON_PATH, "w") as f:
    json.dump(batch_config, f, indent=4)

print(f"Saved {len(mgd_library)} configurations to {BATCH_JSON_PATH}.")


# 5. EXECUTE RF3 BATCH INFERENCE
print("Executing RF3 inference for IKZF1...")
seeds_to_run = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

for current_seed in seeds_to_run:
    print(f"\n--- Starting run for Seed {current_seed} ---")
    
    seed_out_dir = os.path.join(OUTPUT_DIR, f"seed_{current_seed}")
    os.makedirs(seed_out_dir, exist_ok=True)
    
    cmd = [
        "rf3", "fold",
        f"inputs={BATCH_JSON_PATH}",
        f"out_dir={seed_out_dir}",
        "ckpt_path=./rf3_foundry_01_24_latest_remapped.ckpt",
        "early_stopping_plddt_threshold=0.5",
        
        # MODELING OPTIMIZATIONS
        "diffusion_batch_size=1",  # Generate 1 structure per model seed instead of default 5
        "num_steps=50",            # Run 50 diffusion steps instead of default 200
        "n_recycles=5",            # Run 5 cycles through the GNN instead of default 10
        f"seed={current_seed}"     # Model based on 10 different initial model seeds instead of default 1
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Seed {current_seed} inference complete!")
        
    except subprocess.CalledProcessError as e:
        print(f"\nRF3 CRASHED on Seed {current_seed}! Here is the internal error log:")
        print("-" * 40)
        print(e.stderr)
        print("-" * 40)


# 6. EXTRACT FEATURES FOR RIDGE AND CHEMPROP
print("\nExtracting confidence scores...")
extracted_features = []

for cmpd_id, smiles in mgd_library:
    for current_seed in seeds_to_run:
        cmpd_dir = os.path.join(OUTPUT_DIR, f"seed_{current_seed}", str(cmpd_id))
        
        summary_file = os.path.join(cmpd_dir, f"{cmpd_id}_summary_confidences.json")
        score_file = os.path.join(cmpd_dir, f"{cmpd_id}.score")
        
        if os.path.exists(summary_file):
            with open(summary_file, 'r') as f:
                summary = json.load(f)
                
                extracted_features.append({
                    "Compound_ID": cmpd_id,
                    "SMILES": smiles,
                    "Seed": current_seed,
                    "ipTM_ikzf1": summary.get('iptm', None),
                    "pTM_ikzf1": summary.get('ptm', None),
                    "pLDDT_ikzf1": summary.get('overall_plddt', None),
                    "Ranking_Score_ikzf1": summary.get('ranking_score', None),
                    "Early_Stopped_ikzf1": False 
                })
                
        elif os.path.exists(score_file):
            with open(score_file, 'r') as f:
                score_data = json.load(f)
                extracted_features.append({
                    "Compound_ID": cmpd_id,
                    "SMILES": smiles,
                    "Seed": current_seed,
                    "ipTM_ikzf1": 0.0, 
                    "pTM_ikzf1": 0.0,
                    "pLDDT_ikzf1": score_data.get('overall_plddt', None), 
                    "Ranking_Score_ikzf1": 0.0,
                    "Early_Stopped_ikzf1": True
                })
                
        else:
            extracted_features.append({
                "Compound_ID": cmpd_id,
                "SMILES": smiles,
                "Seed": current_seed,
                "ipTM_ikzf1": None,
                "pTM_ikzf1": None,
                "pLDDT_ikzf1": None,
                "Ranking_Score_ikzf1": None,
                "Early_Stopped_ikzf1": None
            })

features_df = pd.DataFrame(extracted_features)
total_trajectories = len(features_df)
successes = len(features_df[features_df['Early_Stopped_ikzf1'] == False])
print(f"Total Trajectories Processed: {total_trajectories} ({len(mgd_library)} compounds x {len(seeds_to_run)} seeds)")
print(f"Successful Binders (Full Fold): {successes}")


# 7. VALIDATE DEGRON-LIGAND DISTANCE IN ALL TRAJECTORIES
print("\nValidating Degron-Ligand distances across all generated models...")

def check_degron_ligand_distance_cif(cif_path, degron_chain='B', ligand_resname='L:0'):
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure('complex', cif_path)
    except Exception as e:
        return None, f"Parse Error: {e}"

    degron_coords = []
    ligand_coords = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if chain.id == degron_chain:
                    for atom in residue:
                        degron_coords.append(atom.coord)
                
                if residue.resname == ligand_resname:
                    for atom in residue:
                        if atom.element != 'ZN':
                            ligand_coords.append(atom.coord)

    if not degron_coords:
        return None, f"Degron chain '{degron_chain}' not found"
    if not ligand_coords:
        return None, f"Ligand '{ligand_resname}' not found"

    degron_coords = np.array(degron_coords)
    ligand_coords = np.array(ligand_coords)

    diff = degron_coords[:, np.newaxis, :] - ligand_coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff**2, axis=2))
    
    min_dist = np.min(dist_matrix)
    return min_dist, "Success"


validation_results = []
for _, row in features_df.iterrows():
    cmpd_id = row['Compound_ID']
    seed = row['Seed']
    
    cif_file = os.path.join(OUTPUT_DIR, f"seed_{seed}", str(cmpd_id), f"{cmpd_id}_model.cif")
    
    if os.path.exists(cif_file):
        dist, status = check_degron_ligand_distance_cif(cif_file)
        validation_results.append({
            "Compound_ID": cmpd_id,
            "Seed": seed,
            "Min_Dist_Angstrom_ikzf1": dist,
            "Dist_Status_ikzf1": status,
            "Flagged_Greater_Than_10A_ikzf1": (dist > 10.0) if dist is not None else True 
        })
    else:
        validation_results.append({
            "Compound_ID": cmpd_id,
            "Seed": seed,
            "Min_Dist_Angstrom_ikzf1": None,
            "Dist_Status_ikzf1": "File not found",
            "Flagged_Greater_Than_10A_ikzf1": True
        })

dist_df = pd.DataFrame(validation_results)

final_combined_df = pd.merge(features_df, dist_df, on=['Compound_ID', 'Seed'], how='left')

successful_complexes = final_combined_df[final_combined_df['Flagged_Greater_Than_10A_ikzf1'] == False]
failed_complexes = final_combined_df[final_combined_df['Flagged_Greater_Than_10A_ikzf1'] == True]

print(f"\nDistance Check Summary:")
print(f"{len(successful_complexes)} trajectories successfully bridged the interface (< 10A).")
print(f"{len(failed_complexes)} trajectories failed to recruit the degron (> 10A).")


# 8. CALCULATE SUCCESS FRACTION PER COMPOUND & MERGE
print("\nCalculating success fractions for each Compound_ID...")

success_fraction_df = final_combined_df.groupby('Compound_ID')['Flagged_Greater_Than_10A_ikzf1'].apply(
    lambda x: (x == False).mean()
).reset_index()

success_fraction_df.rename(columns={'Flagged_Greater_Than_10A_ikzf1': 'Fraction_Success_Under_10A_ikzf1'}, inplace=True)

summary_output_csv = "rf3_compound_success_fractions_ikzf1.csv"
success_fraction_df.to_csv(summary_output_csv, index=False)

print(f"\nFraction of successful complexes (< 10A) per Compound_ID:")
print(success_fraction_df.head())
print(f"Compound summary saved to: {summary_output_csv}")

final_combined_df = pd.merge(final_combined_df, success_fraction_df, on='Compound_ID', how='left')

final_output_csv = "rf3_all_features_and_distances_ikzf1.csv"
final_combined_df.to_csv(final_output_csv, index=False)

print(f"Combined data (with success fractions appended) saved to: {final_output_csv}")
print(f"\nPipeline complete! All features and validations successfully saved.")

Loading molecules from mgd_list.csv...
Successfully loaded 88 compounds.
Saved 88 configurations to mgd_batch_ikzf1.json.
Executing RF3 inference for IKZF1...

Extracting confidence scores...
Total Trajectories Processed: 880 (88 compounds x 10 seeds)
Successful Binders (Full Fold): 880

Validating Degron-Ligand distances across all generated models...

Distance Check Summary:
844 trajectories successfully bridged the interface (< 10A).
36 trajectories failed to recruit the degron (> 10A).

Calculating success fractions for each Compound_ID...

Fraction of successful complexes (< 10A) per Compound_ID:
    Compound_ID  Fraction_Success_Under_10A_ikzf1
0    ALV-05-184                               1.0
1  BML-01-020-L                               1.0
2  BML-01-020-U                               1.0
3    JWJ-01-200                               1.0
4    JWJ-01-203                               1.0
Compound summary saved to: rf3_compound_success_fractions_ikzf1.csv
Combined data (with suc

### **Model Visualization - CRBN:MGD:IKZF1**

In [15]:
from atomworks.io.utils.visualize import view
from atomworks.io.utils.io_utils import load_any

# 1. Define the path to your generated file
model_path = "./rf3_outputs_ikzf1/seed_42/ALV-05-184/ALV-05-184_model.cif"

# 2. Parse the CIF file into a 3D structural object (AtomArray)
parsed_structure = load_any(model_path)

# 3. Render the interactive 3D view
view(parsed_structure)

AtomArrayStack is not supported; using the first model.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### **Energy Minimization and Scoring - CRBN:MGD:IKZF1**

In [6]:
import os
os.environ["OPENMM_CPU_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import traceback
import pandas as pd
import gemmi
import multiprocessing as mp
from openmm import XmlSerializer, LangevinMiddleIntegrator, Platform, NonbondedForce
from openmm.app import PDBFile, Modeller, ForceField, Simulation, NoCutoff, HBonds, Element
from openmm.unit import kelvin, picosecond, kilojoule_per_mole, angstrom, elementary_charge, nanometer
from openff.toolkit import Molecule
from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem


# ==========================================
# PHASE 1: CPU PREPARATION & SERIALIZATION
# ==========================================
def prep_complex_cpu(task_info):
    index, row = task_info
    cmpd_id = row["Compound_ID"]
    smiles = row["SMILES"]
    seed = row["Seed"]
    
    base_dir = os.path.join("rf3_outputs_ikzf1", f"seed_{seed}", str(cmpd_id))
    cif_path = os.path.join(base_dir, f"{cmpd_id}_model.cif")
    pdb_path = os.path.join(base_dir, f"{cmpd_id}_converted.pdb")
    
    system_xml_path = os.path.join(base_dir, f"{cmpd_id}_system.xml")
    ready_pdb_path = os.path.join(base_dir, f"{cmpd_id}_ready.pdb")
    
    if not os.path.exists(cif_path):
        return index, False

    if os.path.exists(system_xml_path) and os.path.exists(ready_pdb_path):
        return index, True

    try:
        # 1. CONVERT CIF TO PDB
        if not os.path.exists(pdb_path):
            doc = gemmi.cif.read_file(cif_path)
            st = gemmi.make_structure_from_block(doc.sole_block())
            st.write_pdb(pdb_path)
            
        # 2. RENAMING & SEPARATION
        prot_pdb_path = os.path.join(base_dir, f"{cmpd_id}_prot.pdb")
        lig_pdb_path = os.path.join(base_dir, f"{cmpd_id}_lig.pdb")
        standard_res = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 
                        'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 
                        'TYR', 'VAL', 'HOH', 'ACE', 'NME', 'MG', 'CA', 'NA', 'CL', 'K', 'ZN'}
        
        with open(pdb_path, 'r') as f: 
            lines = f.readlines()
            
        with open(prot_pdb_path, 'w') as f_prot, open(lig_pdb_path, 'w') as f_lig:
            for line in lines:
                line = line.replace('L:0', 'LIG')
                if line.startswith(('ATOM', 'HETATM')):
                    resname = line[17:21].strip()
                    if resname in standard_res:
                        f_prot.write(line)
                    else:
                        f_lig.write(line)
                elif line.startswith(('HET', 'HETNAM', 'HETSYN', 'FORMUL', 'CONECT')):
                    if 'LIG' in line: f_lig.write(line)
                elif line.startswith('END'):
                    f_prot.write(line); f_lig.write(line)
                else:
                    f_prot.write(line)

        # 3. PREPARE OPENMM FORCEFIELD & FIX PROTEIN
        forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml', 'implicit/gbn2.xml')
        fixer = PDBFixer(filename=prot_pdb_path)
        zn_elem = Element.getBySymbol('Zn')
        for atom in fixer.topology.atoms():
            if atom.residue.name == 'ZN':
                atom.name = 'Zn'; atom.element = zn_elem

        fixer.findMissingResidues()
        fixer.missingResidues = {} 
        fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms() 

        modeller = Modeller(fixer.topology, fixer.positions)
        modeller.addHydrogens(forcefield, pH=7.0)

        # 4. PROCESS LIGAND
        lig_rdkit = Chem.MolFromPDBFile(lig_pdb_path, sanitize=False)
        frags = Chem.GetMolFrags(lig_rdkit, asMols=True)
        off_molecules = []
        for frag in frags:
            if frag.GetNumHeavyAtoms() > 3:
                template_mol = Chem.MolFromSmiles(smiles)
                frag = AllChem.AssignBondOrdersFromTemplate(template_mol, frag)
                Chem.SanitizeMol(frag)
                frag_h = Chem.AddHs(frag, addCoords=True)
                for atom in frag_h.GetAtoms():
                    mi = Chem.AtomPDBResidueInfo()
                    mi.SetResidueName("LIG"); mi.SetResidueNumber(1)
                    atom.SetMonomerInfo(mi)
                off_molecules.append(Molecule.from_rdkit(frag_h, allow_undefined_stereo=True))

        # 5. REGISTER GENERATOR & CREATE SYSTEM
        generator = SMIRNOFFTemplateGenerator(molecules=off_molecules) 
        forcefield.registerTemplateGenerator(generator.generator)
        for mol in off_molecules:
            modeller.add(mol.to_topology().to_openmm(), mol.conformers[0].to_openmm())

        system = forcefield.createSystem(modeller.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
        
        # 6. SERIALIZE TO DISK (The Magic Step)
        with open(system_xml_path, 'w') as f:
            f.write(XmlSerializer.serialize(system))
        
        with open(ready_pdb_path, 'w') as f:
            PDBFile.writeFile(modeller.topology, modeller.positions, f)

        print(f"🛠️ [{cmpd_id}] CPU Prep Complete! Saved to XML.")
        return index, True
        
    except Exception as e:
        print(f"❌ [{cmpd_id}] CPU PREP FAILED: {e}")
        return index, False


# ==========================================
# PHASE 2: GPU MINIMIZATION
# ==========================================
def minimize_complex_gpu(task_info):
    index, row = task_info
    cmpd_id = row["Compound_ID"]
    seed = row["Seed"]
    
    base_dir = os.path.join("rf3_outputs_ikzf1", f"seed_{seed}", str(cmpd_id))
    system_xml_path = os.path.join(base_dir, f"{cmpd_id}_system.xml")
    ready_pdb_path = os.path.join(base_dir, f"{cmpd_id}_ready.pdb")
    
    if not os.path.exists(system_xml_path) or not os.path.exists(ready_pdb_path):
        return index, None

    try:
        # 1. LOAD PRE-BUILT SYSTEM AND POSITIONS
        with open(system_xml_path, 'r') as f:
            system = XmlSerializer.deserialize(f.read())
        
        pdb = PDBFile(ready_pdb_path)
        
        # 2. SIMULATE AND MINIMIZE
        integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picosecond)
        simulation = Simulation(pdb.topology, system, integrator, Platform.getPlatformByName('CUDA'))
        simulation.context.setPositions(pdb.positions)
        
        simulation.minimizeEnergy(tolerance=10.0*kilojoule_per_mole/nanometer)
        
        state = simulation.context.getState(getEnergy=True)
        minimized_energy = state.getPotentialEnergy().value_in_unit(kilojoule_per_mole)
        
        print(f"✅ [{cmpd_id}] GPU Minimized: {minimized_energy:.2f} kJ/mol")

        # 3. FREE VRAM
        del simulation.context
        del simulation
        import gc; gc.collect()
        
        return index, minimized_energy
        
    except Exception as e:
        print(f"❌ [{cmpd_id}] GPU SPRINT FAILED: {e}")
        return index, None

# ==========================================
# MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    csv_path = "rf3_all_features_and_distances_ikzf1.csv"
    df = pd.read_csv(csv_path)
    if "OpenMM_Minimized_Energy_kJ_mol" not in df.columns:
        df["OpenMM_Minimized_Energy_kJ_mol"] = None

    successful_complexes = df[(df["Early_Stopped_ikzf1"] == False)]
    all_tasks = list(successful_complexes.iterrows())

    # --- EXECUTE PHASE 1 ---
    print(f"\n🚀 PHASE 1: Starting CPU Preparation for {len(all_tasks)} complexes...")
    with mp.Pool(processes=32, maxtasksperchild=1) as pool:
        prep_results = pool.map(prep_complex_cpu, all_tasks)

    ready_tasks = [task for task, result in zip(all_tasks, prep_results) if result[1] == True]

    # --- EXECUTE PHASE 2 ---
    print(f"\n⚡ PHASE 2: Starting GPU Minimization Sprint for {len(ready_tasks)} prepped complexes...")
    with mp.Pool(processes=6) as pool:
        min_results = pool.map(minimize_complex_gpu, ready_tasks)

    # --- SAVE RESULTS ---
    for index, energy in min_results:
        if energy is not None:
            df.at[index, "OpenMM_Minimized_Energy_kJ_mol_ikzf1"] = energy

    output_csv = "rf3_features_with_thermodynamics_ikzf1.csv"
    df.to_csv(output_csv, index=False)
    print(f"\n✨ Pipeline finished! Results saved to {output_csv}")


🚀 PHASE 1: Starting CPU Preparation for 880 complexes...


[21:03:03] Explicit valence for atom # 1 C, 5, is greater than permitted


❌ [JWJ-01-248] CPU PREP FAILED: Explicit valence for atom # 1 C, 5, is greater than permitted


[21:03:03] Explicit valence for atom # 11 F, 2, is greater than permitted


❌ [JWJ-01-222] CPU PREP FAILED: Explicit valence for atom # 11 F, 2, is greater than permitted


...

❌ [JWJ-01-251] CPU PREP FAILED: Explicit valence for atom # 35 C, 5, is greater than permitted


[21:03:04] Explicit valence for atom # 0 O, 3, is greater than permitted
[21:03:04] Explicit valence for atom # 6 C, 5, is greater than permitted


❌ [JWJ-01-218] CPU PREP FAILED: Explicit valence for atom # 0 O, 3, is greater than permitted❌ [JWJ-01-334] CPU PREP FAILED: Explicit valence for atom # 6 C, 5, is greater than permitted



[21:03:04] Explicit valence for atom # 1 C, 5, is greater than permitted


❌ [JWJ-01-245] CPU PREP FAILED: Explicit valence for atom # 1 C, 5, is greater than permitted

⚡ PHASE 2: Starting GPU Minimization Sprint for 853 prepped complexes...
✅ [JWJ-01-238] GPU Minimized: -20134.42 kJ/mol
✅ [JWJ-01-252] GPU Minimized: -20270.70 kJ/mol
✅ [JWJ-01-218] GPU Minimized: -20880.33 kJ/mol
✅ [JWJ-01-203] GPU Minimized: -20482.90 kJ/mol
✅ [JWJ-01-231] GPU Minimized: nan kJ/mol
✅ [JWJ-01-252] GPU Minimized: -20341.39 kJ/mol
✅ [JWJ-01-238] GPU Minimized: -20062.26 kJ/mol
✅ [JWJ-01-218] GPU Minimized: -20752.39 kJ/mol
✅ [JWJ-01-203] GPU Minimized: -20449.55 kJ/mol
✅ [JWJ-01-231] GPU Minimized: nan kJ/mol
✅ [JWJ-01-252] GPU Minimized: -20386.88 kJ/mol
✅ [JWJ-01-203] GPU Minimized: -20657.04 kJ/mol
✅ [JWJ-01-238] GPU Minimized: -20399.54 kJ/mol
✅ [JWJ-01-218] GPU Minimized: -20894.45 kJ/mol
✅ [JWJ-01-231] GPU Minimized: nan kJ/mol
✅ [JWJ-01-203] GPU Minimized: -20433.00 kJ/mol
✅ [JWJ-01-238] GPU Minimized: -19935.38 kJ/mol
✅ [JWJ-01-218] GPU Minimized: -20778.87 kJ/mol
✅ [J